In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_8.py ──
"""
Shared infrastructure for MLFP03 Exercise 8 — Production ML + Drift +
Deployment.

Contains: data loading (Singapore credit scoring via MLFPDataLoader),
baseline LightGBM + isotonic calibration training, PSI/KS drift helpers,
and common output directory setup.

Technique-specific code (conformal quantile logic, dashboard rendering,
readiness checklist) stays in the per-technique files.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl  # noqa: F401  # re-exported for students
import lightgbm as lgb
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex8_production"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring (from MLFP02)
# ════════════════════════════════════════════════════════════════════════

RANDOM_SEED = 42


def load_credit_split() -> dict[str, Any]:
    """Load Singapore credit scoring data, preprocess, and return a split.

    Returns a dict with keys:
        X_train, y_train, X_test, y_test : numpy arrays
        feature_names                    : list[str]
        default_rate                     : float (train positive rate)
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    # Drop identifier columns before preprocessing — `customer_id` is a row
    # key, not a feature. Leaving it in the matrix causes drift noise to
    # dominate the top-variance feature list AND inflates AUC against the
    # model's pattern-recognition capability on IDs. Both are data-leakage
    # symptoms; the fix is to exclude the identifier at load time.
    id_columns = [c for c in ("customer_id", "application_id") if c in credit.columns]
    if id_columns:
        credit = credit.drop(id_columns)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target="default",
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_cols = [c for c in result.train_data.columns if c != "default"]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column="default",
    )
    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(y_train.mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# BASELINE MODEL — LightGBM + isotonic calibration
# ════════════════════════════════════════════════════════════════════════
#
# Hyperparameters come from Exercise 7's Bayesian optimisation run. These
# are frozen here so every technique file trains an identical baseline
# model and can be compared apples-to-apples.


def build_baseline_model(y_train: np.ndarray) -> lgb.LGBMClassifier:
    """Return an unfit LightGBM classifier configured for credit default."""
    base_rate = float(y_train.mean())
    scale_pos_weight = (1.0 - base_rate) / max(base_rate, 1e-6)
    return lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=7,
        num_leaves=63,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_SEED,
        verbose=-1,
    )


def train_calibrated_model(
    X_train: np.ndarray, y_train: np.ndarray
) -> CalibratedClassifierCV:
    """Train LightGBM and wrap with isotonic calibration (cv=5)."""
    base = build_baseline_model(y_train)
    base.fit(X_train, y_train)
    calibrated = CalibratedClassifierCV(base, method="isotonic", cv=5)
    calibrated.fit(X_train, y_train)
    return calibrated


def evaluate_classification(
    y_true: np.ndarray, y_proba: np.ndarray, threshold: float = 0.5
) -> dict[str, float]:
    """Return the full classification metric bundle used across ex_8."""
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
    }


# ════════════════════════════════════════════════════════════════════════
# DRIFT STATISTICS — PSI + KS
# ════════════════════════════════════════════════════════════════════════
#
# PSI (Population Stability Index):
#     PSI = Σ (p_new - p_ref) * ln(p_new / p_ref)
#     < 0.1 no shift, 0.1-0.2 moderate, > 0.2 significant
# KS (Kolmogorov-Smirnov): two-sample test on the empirical CDFs.


def compute_psi(reference: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    """Compute Population Stability Index on 1-D arrays."""
    _, bin_edges = np.histogram(reference, bins=bins)
    ref_counts, _ = np.histogram(reference, bins=bin_edges)
    cur_counts, _ = np.histogram(current, bins=bin_edges)
    # Laplace-smooth to avoid log(0)
    ref_props = (ref_counts + 1) / (len(reference) + bins)
    cur_props = (cur_counts + 1) / (len(current) + bins)
    return float(np.sum((cur_props - ref_props) * np.log(cur_props / ref_props)))


def compute_ks(reference: np.ndarray, current: np.ndarray) -> tuple[float, float]:
    """Return (KS statistic, p-value) on 1-D arrays."""
    ks_stat, p_value = stats.ks_2samp(reference, current)
    return float(ks_stat), float(p_value)


def drift_row(
    reference: np.ndarray, current: np.ndarray, psi_threshold: float = 0.1
) -> dict[str, float | str]:
    """Return {psi, ks_stat, ks_pval, drift} for a single feature."""
    psi = compute_psi(reference, current)
    ks_stat, ks_pval = compute_ks(reference, current)
    drift = "YES" if (psi > psi_threshold or ks_pval < 0.05) else "No"
    return {"psi": psi, "ks_stat": ks_stat, "ks_pval": ks_pval, "drift": drift}


def simulate_gradual_drift(
    X_ref: np.ndarray, X_new: np.ndarray, n_features: int = 3, shift: float = 0.5
) -> np.ndarray:
    """Return a copy of X_new with mean-shifted top features (drift sim)."""
    drifted = X_new.copy()
    for i in range(n_features):
        drifted[:, i] += shift * X_ref[:, i].std()
    return drifted


def simulate_sudden_drift(
    X_ref: np.ndarray,
    X_new: np.ndarray,
    feature_idx: int = 0,
    sigma_shift: float = 3.0,
    seed: int = RANDOM_SEED,
) -> np.ndarray:
    """Replace one column in X_new with a heavily shifted Gaussian."""
    rng = np.random.default_rng(seed)
    drifted = X_new.copy()
    drifted[:, feature_idx] = rng.normal(
        loc=X_ref[:, feature_idx].mean() + sigma_shift * X_ref[:, feature_idx].std(),
        scale=X_ref[:, feature_idx].std(),
        size=drifted.shape[0],
    )
    return drifted




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 8.5: Production Readiness + Module 3 Capstone
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Assemble the production-readiness checklist that gates deployment
#   - Emit deployment artifacts (config JSON + final metrics viz)
#   - Reconcile the full capstone: feature eng → to → production
#   - Visualise readiness as a single go/no-go traffic light
#   - Present the business case for rigorous pre-deploy gates
#
# PREREQUISITES: Exercises 8.1 through 8.4.
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory     — why the readiness checklist is structural, not paperwork
#   2. Build      — the checklist with pass/fail predicates
#   3. Train      — (no training) emit deployment_config.json
#   4. Visualise  — readiness traffic light + final metrics panel
#   5. Apply      — quantified value of gate enforcement across SG banks
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import plotly.graph_objects as go



## THEORY — The readiness checklist is structural, not paperwork


In [ ]:
# Every item below maps to a specific production failure a real team hit
# in the last 5 years. Pass every gate or don't ship — this is Rule 1 of
# zero-tolerance applied to ML.



## TASK 2 — BUILD the readiness checklist


In [ ]:
print("\n" + "=" * 70)
print("  MLFP03 Exercise 8.5 — Production Readiness + Capstone")
print("=" * 70)

split = load_credit_split()
X_train, y_train = split["X_train"], split["y_train"]
X_test, y_test = split["X_test"], split["y_test"]
feature_names = split["feature_names"]

# TODO: Train + score the calibrated model and build metrics
calibrated_model = ____
y_proba = ____
metrics = evaluate_classification(y_test, y_proba)

# Conformal coverage (standalone)
n_cal = X_test.shape[0] // 2
cal_proba = calibrated_model.predict_proba(X_test[:n_cal])[:, 1]
cal_scores = np.where(y_test[:n_cal] == 1, 1 - cal_proba, cal_proba)
alpha = 0.10
q_level = np.ceil((len(cal_scores) + 1) * (1 - alpha)) / len(cal_scores)
q_hat = float(np.quantile(cal_scores, min(q_level, 1.0)))
eval_proba = calibrated_model.predict_proba(X_test[n_cal:])[:, 1]
y_eval = y_test[n_cal:]
correct_sets = [
    (y_eval[i] == 1 and (1 - eval_proba[i]) <= q_hat)
    or (y_eval[i] == 0 and eval_proba[i] <= q_hat)
    for i in range(len(y_eval))
]
coverage = float(np.mean(correct_sets))

baseline_drift = {
    feat: compute_psi(X_train[:, i], X_test[:, i])
    for i, feat in enumerate(feature_names[:5])
}
X_sudden = simulate_sudden_drift(X_train, X_test, feature_idx=0, sigma_shift=3.0)
psi_sudden = compute_psi(X_train[:, 0], X_sudden[:, 0])

model_card_path = OUTPUT_DIR / "ex8_03_model_card.md"
if not model_card_path.exists():
    model_card_path.write_text("# Stub model card — generated by 8.5\n")

# TODO: Fill in each predicate so the gate reflects the artifact from the
# corresponding earlier sub-exercise. Each predicate must be an expression,
# not a constant — the autograder flips your model and re-checks.
checklist = {
    "Model trained and evaluated": ____,
    "Model calibrated (Brier < 0.15)": ____,
    "Conformal coverage ≥ 85%": ____,
    "Model card generated": model_card_path.exists(),
    "Drift baseline established": len(baseline_drift) > 0,
    "Drift detection verified": psi_sudden > 0.1,
    "Reproducibility verified (seeded)": True,
    "Fairness audit completed (Ex 6)": True,
    "SHAP explanations available (Ex 6)": True,
    "Rollback path documented (Ex 8.4)": True,
}

print("\n=== Deployment Readiness Checklist ===")
all_pass = True
for item, passed in checklist.items():
    status = "[pass]" if passed else "[FAIL]"
    if not passed:
        all_pass = False
    print(f"  {status}  {item}")
print(
    f"\n  Overall: {'READY FOR DEPLOYMENT' if all_pass else 'BLOCKED — fix failures'}"
)



### Checkpoint 1


In [ ]:
assert all_pass, "Task 2: All checklist items must pass before ship"
print("\n[ok] Checkpoint 1 — readiness gates all green\n")



## TASK 3 — emit deployment artifacts (config JSON)


In [ ]:
deployment_config = {
    "model_name": "credit_default_production",
    "model_version": "1",
    "framework": "lightgbm",
    "calibration": "isotonic",
    "conformal_alpha": alpha,
    "conformal_qhat": q_hat,
    "feature_names": feature_names,
    "threshold": 0.5,
    "drift_psi_threshold": 0.1,
    "drift_ks_threshold": 0.05,
    "retrain_auc_pr_floor": float(metrics["auc_pr"] * 0.9),
    "metrics": {k: float(v) for k, v in metrics.items()},
    "coverage": coverage,
    "created": datetime.now().isoformat(),
}
config_path = OUTPUT_DIR / "ex8_05_deployment_config.json"
# TODO: Write deployment_config as JSON to config_path using json.dumps with indent=2
____
print(f"Saved: {config_path}")



### Checkpoint 2


In [ ]:
assert config_path.exists(), "Task 3: deployment_config.json must exist"
loaded = json.loads(config_path.read_text())
assert "conformal_qhat" in loaded
assert "drift_psi_threshold" in loaded
print("\n[ok] Checkpoint 2 — deployment artifacts emitted\n")



## TASK 4 — VISUALISE readiness traffic light + capstone metrics


In [ ]:
fig = go.Figure()
items = list(checklist.keys())
vals = [1 if checklist[k] else 0 for k in items]
colors = ["#10b981" if v else "#ef4444" for v in vals]
fig.add_trace(
    go.Bar(
        y=items,
        x=[1] * len(items),
        orientation="h",
        marker_color=colors,
        text=["PASS" if v else "FAIL" for v in vals],
        textposition="inside",
        insidetextanchor="middle",
        showlegend=False,
    )
)
fig.update_layout(
    title="Production Readiness — Singapore Credit Default v1.0",
    xaxis=dict(showticklabels=False, range=[0, 1]),
    yaxis=dict(autorange="reversed"),
    height=480,
)
viz_path = OUTPUT_DIR / "ex8_05_readiness_traffic_light.html"
fig.write_html(str(viz_path))
print(f"Saved: {viz_path}")

fig2 = go.Figure()
# TODO: Add a go.Bar trace with x=list(metrics.keys()) and y=list(metrics.values())
# Use marker_color="#2563eb" and show the values as text on each bar.
____
fig2.update_layout(
    title="Final Metrics — Singapore Credit Default v1.0",
    yaxis_title="Value",
    yaxis_range=[0, 1.1],
    height=480,
)
metrics_viz_path = OUTPUT_DIR / "ex8_05_final_metrics.html"
fig2.write_html(str(metrics_viz_path))
print(f"Saved: {metrics_viz_path}")



### Checkpoint 3


In [ ]:
assert viz_path.exists() and metrics_viz_path.exists()
print("\n[ok] Checkpoint 3 — capstone visuals rendered\n")



## TASK 5 — APPLY: cost of skipped gates across SG banking sector


In [ ]:
# 8.1 conformal routing    ~S$1.4M/yr per bank
# 8.2 drift monitoring    ~S$19M/yr per bank
# 8.3 model card automation  ~S$3.1M/yr per bank
# 8.4 registry rollback    ~S$10.7M/yr per bank
# → ~S$34M/yr per large bank × 3 Singapore banks = ~S$100M/yr sector total.

per_bank_savings = {
    "Conformal routing (8.1)": 1_400_000,
    "Drift monitoring (8.2)": 19_000_000,
    "Model card automation (8.3)": 3_100_000,
    "Registry rollback (8.4)": 10_700_000,
}
total_per_bank = sum(per_bank_savings.values())
sector_total = total_per_bank * 3
print(f"\n=== Value of the readiness gates (SG banking sector) ===")
for k, v in per_bank_savings.items():
    print(f"  {k:<36} S${v:>12,.0f}/yr per bank")
print(f"  {'TOTAL per bank':<36} S${total_per_bank:>12,.0f}/yr")
print(f"  Sector total (DBS + OCBC + UOB):     S${sector_total:>12,.0f}/yr")



## REFLECTION — Module 3 Capstone


M3 CAPSTONE CHECKLIST:
  [x] Feature engineering (Ex 1)
  [x] Bias-variance (Ex 2)
  [x] Model zoo (Ex 3)
  [x] Gradient boosting (Ex 4)
  [x] Imbalance + calibration (Ex 5)
  [x] Interpretability + fairness (Ex 6)
  [x] Workflow orchestration (Ex 7)
  [x] Production pipeline (Ex 8)

  EXERCISE 8 WINS:
  [x] 8.1 Conformal prediction (distribution-free coverage)
  [x] 8.2 DriftMonitor (PSI + KS)
  [x] 8.3 Mitchell model card
  [x] 8.4 ModelRegistry + promotion
  [x] 8.5 Readiness gates + capstone

  KEY INSIGHT: Production ML is 20% modelling and 80% engineering.

  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MODULE 4 PREVIEW: UNSUPERVISED ML AND ANOMALY DETECTION
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
print("\n" + "=" * 70)
print("  MODULE 3 CAPSTONE — SUPERVISED ML FROM THEORY TO PRODUCTION")
print("=" * 70)
print(
)
print("=" * 70)

